## Train the LightGBM model

### 1. Hyperparameter tuning

Each Optuna trial picks a value for every hyperparameter, trains the model on the 3 validation folds,
and returns the mean Poisson deviance (lower is better). <br> Optuna uses the results to pick better values in the next trials.

The test set is not used here.

In [ ]:
# this code was needed to run from the directory on GitHub using Colab

# 1. get the repo + the PR branch (the data is already in it)
!git clone -q https://github.com/amelia-lee-smu/CS610-Applied-Machine-Learning.git
%cd CS610-Applied-Machine-Learning
!git fetch -q origin pull/6/head:pr6 && git checkout -q pr6

# 2. add the LightGBM model file and the seed constant
!cp /content/lightgbm_model.py utilities/
!printf '\n# fixed seed for the models themselves\nRANDOM_SEED = 42\n' >> utilities/values.py

# 3. packages (Colab already has lightgbm, pandas, scikit-learn, pyarrow)
!pip install -q optuna

# 4. the notebook expects to run from the notebooks/ folder
import os
os.chdir('/content/CS610-Applied-Machine-Learning/notebooks')

/content/CS610-Applied-Machine-Learning/notebooks/CS610-Applied-Machine-Learning


In [ ]:
import sys
from pathlib import Path

import optuna

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utilities.lightgbm_model import *
from utilities.training import *
from utilities.utilities import *
from utilities.values import *

data_dir = '../data/'
model_dir = '../models/'

# read the train-validation set (Jan 2020 - Feb 2021)
train_validation_df = read_parquet_file(data_dir + 'processed/train_validation.parquet')

In [ ]:
def objective(trial):
    # 1. let optuna pick the hyperparams for this trial
    hyperparams = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log = True),      # search on a log scale
        'n_estimators': trial.suggest_int('n_estimators', 100, 800, step = 50),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255, log = True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 300, log = True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log = True),
    }

    # 2. train a new model with these hyperparams on each validation fold, and score it
    scores_df = cross_validate(LightGBMModel, hyperparams, train_validation_df)
    mean_scores = scores_df[['MAE', 'RMSE', 'poisson_deviance']].mean()

    # Optuna tunes on the Poisson deviance only
    # MAE and RMSE are stored with the trial so they also appear in the results table
    trial.set_user_attr('MAE', mean_scores['MAE'])
    trial.set_user_attr('RMSE', mean_scores['RMSE'])

    # show the scores of this trial
    print('Poisson deviance per fold:', scores_df['poisson_deviance'].round(4).tolist())
    print(f'Trial {trial.number}: mean Poisson deviance = {mean_scores["poisson_deviance"]:.4f}, '
          f'mean MAE = {mean_scores["MAE"]:.4f}, mean RMSE = {mean_scores["RMSE"]:.4f}\n')
    return mean_scores['poisson_deviance']


# hide Optuna's own log line for every trial (the progress bar below shows the progress instead)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# the seed makes the search repeatable
study = optuna.create_study(direction = 'minimize', sampler = optuna.samplers.TPESampler(seed = OPTUNA_SEED))

In [ ]:
n_trials = 30

# the progress bar shows how many trials are done, the time remaining and the best score so far
study.optimize(objective, n_trials = n_trials, show_progress_bar = True)

print('Best hyperparameters:', study.best_params)
print('Best mean Poisson deviance:', round(study.best_value, 4))
print('Mean MAE of the best trial:', round(study.best_trial.user_attrs['MAE'], 4))
print('Mean RMSE of the best trial:', round(study.best_trial.user_attrs['RMSE'], 4))

  0%|          | 0/30 [00:00<?, ?it/s]

Fold 1: training LightGBMModel on 298605 rows
Fold 2: training LightGBMModel on 373269 rows
Fold 3: training LightGBMModel on 447933 rows
Poisson deviance per fold: [1.0616, 0.9234, 0.6613]
Trial 0: mean Poisson deviance = 0.8821, mean MAE = 0.5466, mean RMSE = 0.9680

Fold 1: training LightGBMModel on 298605 rows
Fold 2: training LightGBMModel on 373269 rows
Fold 3: training LightGBMModel on 447933 rows
Poisson deviance per fold: [1.0732, 0.949, 0.696]
Trial 1: mean Poisson deviance = 0.9061, mean MAE = 0.5599, mean RMSE = 1.1572

Fold 1: training LightGBMModel on 298605 rows
Fold 2: training LightGBMModel on 373269 rows
Fold 3: training LightGBMModel on 447933 rows
Poisson deviance per fold: [1.2347, 1.0434, 0.8604]
Trial 2: mean Poisson deviance = 1.0462, mean MAE = 0.6324, mean RMSE = 0.9531

Fold 1: training LightGBMModel on 298605 rows
Fold 2: training LightGBMModel on 373269 rows
Fold 3: training LightGBMModel on 447933 rows
Poisson deviance per fold: [1.1973, 1.0073, 0.8003]
Tr

In [ ]:
# all trials, best first
trials_df = study.trials_dataframe()
trials_df.sort_values('value').head(10)

,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_min_child_samples,params_n_estimators,params_num_leaves,params_reg_lambda,params_subsample,user_attrs_MAE,user_attrs_RMSE,system_attrs_tpe:relative_params:0,state
29,29,0.868933,2026-10-30 04:01:09.944171,2026-10-30 04:03:59.162586,0 days 00:02:49.218415,0.538421,0.042490,38,700,230,0.048416,0.836855,0.532315,0.938555,"{""colsample_bytree"": 0.5384209448817046, ""lear...",COMPLETE
28,28,0.870978,2026-10-30 03:58:13.733838,2026-10-30 04:01:09.938320,0 days 00:02:56.204482,0.507050,0.025402,32,700,220,0.008892,0.728937,0.535670,0.931947,"{""colsample_bytree"": 0.5070502458908716, ""lear...",COMPLETE
22,22,0.872011,2026-10-30 03:42:34.655816,2026-10-30 03:45:17.700231,0 days 00:02:43.044415,0.512163,0.037522,51,700,175,0.009678,0.739294,0.537161,0.942122,"{""colsample_bytree"": 0.5121634269746602, ""lear...",COMPLETE
24,24,0.872553,2026-10-30 03:48:00.855271,2026-10-30 03:50:56.350038,0 days 00:02:55.494767,0.509984,0.062546,70,800,190,0.020959,0.865927,0.532166,0.953478,"{""colsample_bytree"": 0.5099842003104023, ""lear...",COMPLETE
19,19,0.875332,2026-10-30 03:33:44.901480,2026-10-30 03:36:53.633321,0 days 00:03:08.731841,0.524076,0.076654,65,800,236,0.003291,0.743097,0.531916,0.957046,"{""colsample_bytree"": 0.5240757428171076, ""lear...",COMPLETE
25,25,0.876329,2026-10-30 03:50:56.354133,2026-10-30 03:53:39.820583,0 days 00:02:43.466450,0.531353,0.084336,46,750,230,0.005652,0.886633,0.530007,0.954481,"{""colsample_bytree"": 0.5313532137399997, ""lear...",COMPLETE
27,27,0.876433,2026-10-30 03:55:53.293930,2026-10-30 03:58:13.731058,0 days 00:02:20.437128,0.597838,0.065883,66,600,226,0.010130,0.800040,0.537697,0.967089,"{""colsample_bytree"": 0.5978379743017761, ""lear...",COMPLETE
21,21,0.876529,2026-10-30 03:39:36.126474,2026-10-30 03:42:34.650481,0 days 00:02:58.524007,0.528189,0.059216,137,750,195,0.003198,0.736381,0.535863,0.961487,"{""colsample_bytree"": 0.5281891662038731, ""lear...",COMPLETE
17,17,0.878167,2026-10-30 03:29:59.820840,2026-10-30 03:32:09.624362,0 days 00:02:09.803522,0.651282,0.045009,53,550,200,0.001274,0.766648,0.543248,0.963638,"{""colsample_bytree"": 0.6512822187064345, ""lear...",COMPLETE
13,13,0.878660,2026-10-30 03:21:25.344333,2026-10-30 03:23:50.941215,0 days 00:02:25.596882,0.552709,0.094720,45,600,233,0.002824,0.637949,0.530141,0.948011,"{""colsample_bytree"": 0.5527091571221081, ""lear...",COMPLETE


### 2. Validation predictions (for the ensemble)

- Get the best set of hyperparameters from optuna.
- For each of the cross-validation sets, train the model on the train set, and store the validation set predictions.
- These validation set predictions will be used to train the ensemble model later.

In [ ]:
# get the predictions of each validation fold
fold_results = get_out_of_fold_predictions(LightGBMModel, study.best_params, train_validation_df)

# save the predictions in each file separately
for fold_number, fold_df in enumerate(fold_results, start = 1):
    save_to_parquet_file(fold_df, data_dir + f'predictions/validation_lightgbm_fold{fold_number}.parquet')
print('Validation predictions saved.')

Fold 1: training LightGBMModel on 298605 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Fold 2: training LightGBMModel on 373269 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Fold 3: training LightGBMModel on 447933 rows


/content/CS610-Applied-Machine-Learning/utilities/training.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df[PREDICTION] = model.predict(validation_df)
/content/CS610-Applied-Machine-Learning/utilities/training.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  validation_df['fold'] = fold_number


Validation predictions saved.


### 3. Final model and test set prediction

In [ ]:
test_df = read_parquet_file(data_dir + 'processed/test.parquet')

In [ ]:
# train a new model on the entire train set (Jan 2020 - Feb 2021) using the best hyperparams from Optuna
model = LightGBMModel(**study.best_params)
model.fit(train_validation_df)

# do predictions on the test set (Mar -April 2021)
test_df[PREDICTION] = model.predict(test_df)

# save the model and results
model.save_model(model_dir + 'lightgbm_model.pkl')
save_to_parquet_file(test_df, data_dir + 'predictions/test_lightgbm.parquet')
print('Model and test predictions saved.')

Model and test predictions saved.


In [ ]:
# test set scores (for reporting only: the test set was not used to choose any hyperparameter)
test_scores = evaluate(model, test_df)
print({name: round(score, 4) for name, score in test_scores.items()})

{'MAE': 0.5099, 'RMSE': 0.9865, 'poisson_deviance': 1.009}


In [ ]:
test_df.head()

,date,num_rentals,start station name,hour_of_day,day_of_week,num_rentals_t-2,num_rentals_t-3,sine_week_of_year,cosine_week_of_year,temperature_2m,snow_depth,cloud_cover,cloud_cover_mid,relative_humidity_2m,precipitation,wind_speed_10m,is_public_holiday,prediction
0,2021-03-01,0,5 Corners Library,0,0,0.0,0.0,0.812487,0.582979,5.55,0.1,100.0,99.0,96.586780,0.5,10.787993,0,0.019777
1,2021-03-01,0,5 Corners Library,1,0,0.0,0.0,0.812487,0.582979,5.40,0.1,100.0,100.0,97.596886,0.7,12.181624,0,0.009886
2,2021-03-01,0,5 Corners Library,2,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.4,9.164103,0,0.006132
3,2021-03-01,0,5 Corners Library,3,0,0.0,0.0,0.812487,0.582979,5.45,0.1,100.0,100.0,97.597850,0.5,6.952755,0,0.004384
4,2021-03-01,0,5 Corners Library,4,0,0.0,0.0,0.812487,0.582979,5.05,0.1,100.0,100.0,98.273410,1.0,8.121970,0,0.005624


In [ ]:
!cd .. && zip -r lightgbm_results.zip models/lightgbm_model.pkl data/predictions/*lightgbm*.parquet
from google.colab import files
files.download('../lightgbm_results.zip')

  adding: models/lightgbm_model.pkl (deflated 57%)
  adding: data/predictions/test_lightgbm.parquet (deflated 28%)
  adding: data/predictions/validation_lightgbm_fold1.parquet (deflated 12%)
  adding: data/predictions/validation_lightgbm_fold2.parquet (deflated 16%)
  adding: data/predictions/validation_lightgbm_fold3.parquet (deflated 15%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>